# 16 Second Cohort

**Purpose.** Second cohort (different instrument): build profiles, test acquisition-date batch confound.

**Thesis reference.** §5 A Second Cohort

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

In [1]:
import re, json, os
from pathlib import Path
from collections import Counter
import numpy as np, pyopenms as oms

ROOT    = Path("../second_cohort_mzml")
OUT     = Path("second_cohort"); OUT.mkdir(exist_ok=True)
MZ_MIN, MZ_MAX, RES = 50.0, 1650.0, 0.02
NBINS   = int(round((MZ_MAX - MZ_MIN) / RES))
POLARITY = "Pos"

files, meta = [], {}
for d in sorted(ROOT.glob(f"mzML-*{POLARITY}")):
    stage = 0 if "NoCKD" in d.name else int(re.search(r"CKD(\d)", d.name).group(1))
    dia   = "nonDiabetes" not in d.name
    for f in sorted(d.glob("*.mzML")):
        m = re.search(r"(P\d{4})", f.name)
        pid = m.group(1) if m else f.stem
        files.append((f, pid, stage, dia))

print(f"{len(files)} files, {len({p for _,p,_,_ in files})} distinct patient ids")
print("  by stage:", dict(sorted(Counter(s for *_ ,s,_ in [(a,b,c,d) for a,b,c,d in files]).items())))

# overlap with the existing cohort
try:
    known = set(json.load(open("profiles/profile_pids.json")))
    ids   = {p for _,p,_,_ in files}
    print(f"\noverlap with the existing cohort: {len(ids & known)} of {len(ids)}")
    print(f"  examples: {sorted(ids & known)[:6]}")
except FileNotFoundError:
    known = set(); print("\n(profiles/profile_pids.json not found; skipping overlap)")

# build the profiles
X, pids, kept = [], [], []
for k,(f,pid,stage,dia) in enumerate(files,1):
    try:
        e = oms.MSExperiment(); oms.MzMLFile().load(str(f), e)
        mz, inty = np.concatenate([s.get_peaks()[0] for s in e.getSpectra()]), \
                   np.concatenate([s.get_peaks()[1] for s in e.getSpectra()])
        m = (mz >= MZ_MIN) & (mz < MZ_MAX)
        prof, _ = np.histogram(mz[m], bins=NBINS, range=(MZ_MIN, MZ_MAX), weights=inty[m])
        X.append(prof.astype(np.float32)); pids.append(pid)
        kept.append(dict(pid=pid, stage=stage, diabetes=dia, peaks=int(m.sum())))
    except Exception as ex:
        print(f"  {f.name}: FAILED {type(ex).__name__}")
    if k % 50 == 0: print(f"  {k}/{len(files)}")

X = np.vstack(X)
np.save(OUT/"mz_profiles.npy", X)
json.dump(pids, open(OUT/"profile_pids.json","w"))
json.dump(kept, open(OUT/"metadata.json","w"), indent=1)
print(f"\nprofiles: {X.shape}   nonzero bins per patient (median): "
      f"{np.median((X>0).sum(1)):.0f} of {NBINS}")

# label files
os.makedirs("labels_second_cohort", exist_ok=True)
M = {r["pid"]: r for r in kept}
def write(name, fn):
    d = {p:v for p,r in M.items() if (v:=fn(r)) is not None}
    if len(set(d.values())) > 1:
        json.dump({"task":name,"labels":d}, open(f"labels_second_cohort/labels_{name}.json","w"), indent=1)
        print(f"  {name:22s} {len(d):4d} patients, {sum(d.values()):4d} positive")
print("\nlabel files:")
write("detect_any_ckd", lambda r: 1 if r["stage"]>0 else 0)
write("progression",    lambda r: 1 if r["stage"]>=3 else 0)
write("severity",       lambda r: (1 if r["stage"]>=3 else 0) if r["stage"]>0 else None)
write("stage_1_vs_2",   lambda r: (1 if r["stage"]==2 else 0) if r["stage"] in (1,2) else None)

222 files, 222 distinct patient ids
  by stage: {0: 37, 1: 28, 2: 80, 3: 70, 4: 7}

overlap with the existing cohort: 90 of 222
  examples: ['P0004', 'P0010', 'P0012', 'P0083', 'P0103', 'P0121']
  50/222
  100/222
  150/222
  200/222

profiles: (222, 80000)   nonzero bins per patient (median): 659 of 80000

label files:
  detect_any_ckd          222 patients,  185 positive
  progression             222 patients,   77 positive
  severity                185 patients,   77 positive
  stage_1_vs_2            108 patients,   80 positive


In [2]:

#   1. does the acquisition-order confound replicate?
#   2. does the severity signal replicate?
#   3. do the two instruments agree on the 90 shared patients?

import json
import numpy as np
from pathlib import Path
from scipy.stats import mannwhitneyu, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold

E = Path("second_cohort")
Xe   = np.load(E/"mz_profiles.npy")
pe   = json.load(open(E/"profile_pids.json"))
print(f"second_cohort: {Xe.shape[0]} patients x {Xe.shape[1]} bins")

# keep bins present in at least 10% of patients --- variance ranking is
# useless when 99% of bins are empty
occ  = (Xe > 0).mean(0)
keep = np.where(occ >= 0.10)[0]
print(f"bins present in >=10% of patients: {len(keep)}")
Xk = Xe[:, keep]
t = Xk.sum(1, keepdims=True); t[t==0] = 1
Xk = np.log1p(Xk / t * 1e6)

mk = lambda: Pipeline([("s", StandardScaler()),
                       ("c", LogisticRegression(C=0.1, max_iter=3000,
                                                class_weight="balanced"))])

def evaluate(name, lab):
    idx = [i for i,p in enumerate(pe) if p in lab]
    y   = np.array([lab[pe[i]] for i in idx])
    if len(np.unique(y)) < 2 or min(np.bincount(y)) < 8:
        print(f"  {name:18s} too few in one class"); return
    X = Xk[idx]
    ids = np.array([int("".join(c for c in pe[i] if c.isdigit())) for i in idx])
    u,_ = mannwhitneyu(ids[y==1], ids[y==0])
    a = u/((y==1).sum()*(y==0).sum()); id_auc = max(a, 1-a)
    oof = np.zeros(len(y)); cnt = np.zeros(len(y))
    for tr,te in RepeatedStratifiedKFold(n_splits=5, n_repeats=10,
                                         random_state=42).split(X,y):
        oof[te] += mk().fit(X[tr],y[tr]).predict_proba(X[te])[:,1]; cnt[te]+=1
    auc = roc_auc_score(y, oof/cnt)
    rng = np.random.default_rng(0); bs=[]
    o = oof/cnt
    for _ in range(2000):
        i = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[i]))>1: bs.append(roc_auc_score(y[i], o[i]))
    lo,hi = np.percentile(bs,[2.5,97.5])
    print(f"  {name:18s} n={len(y):4d} ({int(y.sum()):3d}/{int((1-y).sum()):3d})  "
          f"AUC {auc:.4f} [{lo:.3f},{hi:.3f}]  ID {id_auc:.4f}  "
          f"margin {auc-id_auc:+.4f}  {'BEATS ID' if lo>id_auc else ''}")

print("\n1 & 2. TASKS ON THE SECOND COHORT")
print(f"  {'task':18s} {'n':>10s}  results")
for lf in sorted(Path("labels_second_cohort").glob("labels_*.json")):
    evaluate(lf.stem.replace("labels_",""), json.load(open(lf))["labels"])

# 3. instrument agreement
print("\n3. AGREEMENT BETWEEN THE TWO INSTRUMENTS")
try:
    Xo = np.load("profiles/mz_profiles.npy")
    po = json.load(open("profiles/profile_pids.json"))
    shared = sorted(set(pe) & set(po))
    print(f"  {len(shared)} shared patients")
    ie = {p:i for i,p in enumerate(pe)}; io = {p:i for i,p in enumerate(po)}
    # compare on the bins that are populated in both
    occ_o = (Xo > 0).mean(0)
    both = np.where((occ >= 0.10) & (occ_o >= 0.10))[0]
    print(f"  bins populated in both: {len(both)}")
    if len(both) > 20:
        A = np.log1p(Xe[[ie[p] for p in shared]][:, both])
        B = np.log1p(Xo[[io[p] for p in shared]][:, both])
        # per-patient correlation between the two instruments
        rs = [spearmanr(A[i], B[i])[0] for i in range(len(shared))]
        print(f"  per-patient correlation across instruments: "
              f"median {np.median(rs):.3f}  range [{np.min(rs):.3f}, {np.max(rs):.3f}]")
        # does a patient look more like themselves than like others?
        C = np.corrcoef(np.vstack([A,B]))[:len(shared), len(shared):]
        self_r  = np.diag(C)
        other_r = C[~np.eye(len(shared),dtype=bool)]
        rank = [(C[i] > C[i,i]).sum() for i in range(len(shared))]
        print(f"  self-similarity  {np.median(self_r):.3f}   "
              f"other-similarity {np.median(other_r):.3f}")
        print(f"  patients whose own second measurement is the closest match: "
              f"{sum(r==0 for r in rank)}/{len(shared)}")
        print("\n  A high self-similarity relative to others means the two")
        print("  instruments agree about who each patient is. A low one means")
        print("  the measurements are not comparable at the profile level.")
except FileNotFoundError:
    print("  original profiles not found; skipping")

second cohort: 222 patients x 80000 bins
bins present in >=10% of patients: 1909

1 & 2. TASKS ON THE SECOND COHORT'S DATA
  task                        n  results
  detect_any_ckd     n= 222 (185/ 37)  AUC 1.0000 [1.000,1.000]  ID 0.8711  margin +0.1289  BEATS ID
  progression        n= 222 ( 77/145)  AUC 0.9931 [0.985,0.998]  ID 0.7764  margin +0.2167  BEATS ID
  severity           n= 185 ( 77/108)  AUC 0.9916 [0.983,0.998]  ID 0.6999  margin +0.2917  BEATS ID
  stage_1_vs_2       n= 108 ( 80/ 28)  AUC 1.0000 [1.000,1.000]  ID 1.0000  margin +0.0000  

3. AGREEMENT BETWEEN THE TWO INSTRUMENTS
  90 shared patients
  bins populated in both: 1909
  per-patient correlation across instruments: median 0.090  range [-0.229, 0.246]
  self-similarity  0.057   other-similarity 0.057
  patients whose own second measurement is the closest match: 2/90

  A high self-similarity relative to others means the two
  instruments agree about who each patient is. A low one means
  the measurements are no

In [3]:
import re, json
from pathlib import Path
from collections import defaultdict
import numpy as np

ROOT = Path("../second_cohort_mzml")

by = defaultdict(list)
allnames = []
for d in sorted(ROOT.glob("mzML-*Pos")):
    g = d.name.replace("mzML-Diabetes-","").replace("-Pos","")
    for f in sorted(d.glob("*.mzML")):
        m = re.search(r"P(\d{4})", f.name)
        if m: by[g].append(int(m.group(1)))
        allnames.append((g, f.name))

print("ID RANGE PER GROUP")
print(f"  {'group':8s} {'n':>4s} {'min':>6s} {'max':>6s}   overlap with other groups")
spans = {}
for g, v in sorted(by.items()):
    v = sorted(v); spans[g] = (min(v), max(v))
    print(f"  {g:8s} {len(v):4d} {min(v):6d} {max(v):6d}")

print("\n  do the ranges overlap?")
gs = sorted(spans)
overlaps = 0
for i,a in enumerate(gs):
    for b in gs[i+1:]:
        lo = max(spans[a][0], spans[b][0]); hi = min(spans[a][1], spans[b][1])
        if lo <= hi:
            overlaps += 1
            print(f"    {a} and {b} overlap in {lo}--{hi}")
if overlaps == 0:
    print("    NO OVERLAP AT ALL --- the identifiers are assigned by group,")
    print("    not by enrollment. They encode the label and cannot serve as")
    print("    an acquisition-order baseline.")

print("\nFILENAME STRUCTURE (are there other fields?)")
for g, n in allnames[:8]:
    print(f"  {g:8s} {n}")

print("\nIs there a date in the filename? That would be the real acquisition order.")
dates = set()
for g, n in allnames:
    m = re.match(r"(\d{1,2}-\d{1,2}-\d{4})", n)
    if m: dates.add(m.group(1))
print(f"  {len(dates)} distinct dates: {sorted(dates)[:8]}")
if len(dates) > 1:
    print("  -> use the date as the acquisition-order proxy instead of P-number")
    per = defaultdict(set)
    for g, n in allnames:
        m = re.match(r"(\d{1,2}-\d{1,2}-\d{4})", n)
        if m: per[g].add(m.group(1))
    print("\n  dates per group:")
    for g in sorted(per): print(f"    {g:8s} {sorted(per[g])}")
    print("\n  If each group was run on its own date, group and batch are")
    print("  completely confounded and no classification result from this")
    print("  data can be interpreted.")

ID RANGE PER GROUP
  group       n    min    max   overlap with other groups
  CKD2       80      4   1026
  CKD3       70     12   1027
  CKD4        7    222    760

  do the ranges overlap?
    CKD2 and CKD3 overlap in 12--1026
    CKD2 and CKD4 overlap in 222--760
    CKD3 and CKD4 overlap in 222--760

FILENAME STRUCTURE (are there other fields?)
  CKD1     4-3-2025_0045_38_1_290.mzML.mzML
  CKD1     4-3-2025_0098_39_1_292.mzML.mzML
  CKD1     4-3-2025_0110_40_1_294.mzML.mzML
  CKD1     4-3-2025_0210_41_1_296.mzML.mzML
  CKD1     4-3-2025_0279_42_1_298.mzML.mzML
  CKD1     4-3-2025_0309_43_1_300.mzML.mzML
  CKD1     4-3-2025_0332_44_1_302.mzML.mzML
  CKD1     4-3-2025_0375_45_1_304.mzML.mzML

Is there a date in the filename? That would be the real acquisition order.
  8 distinct dates: ['4-1-2025', '4-2-2025', '4-3-2025', '4-4-2025', '4-5-2025', '4-6-2025', '4-7-2025', '4-8-2025']
  -> use the date as the acquisition-order proxy instead of P-number

  dates per group:
    CKD1     

In [4]:
import re, json, os
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np, pyopenms as oms

ROOT = Path("../second_cohort_mzml")
OUT  = Path("second_cohort"); OUT.mkdir(exist_ok=True)
MZ_MIN, MZ_MAX, RES = 50.0, 1650.0, 0.02
NBINS = int(round((MZ_MAX-MZ_MIN)/RES))

# filename: <date>_<patient>_<seq>_<x>_<y>.mzML.mzML
# the patient field is 4 digits, with or without a leading P
PAT = re.compile(r"^(\d{1,2}-\d{1,2}-\d{4})_P?(\d{4})_(\d+)_")

rows = []
for d in sorted(ROOT.glob("mzML-*Pos")):
    stage = 0 if "NoCKD" in d.name else int(re.search(r"CKD(\d)", d.name).group(1))
    for f in sorted(d.glob("*.mzML")):
        m = PAT.match(f.name)
        if not m:
            print(f"  unparsed: {f.name}"); continue
        date, pid, seq = m.group(1), f"P{int(m.group(2)):04d}", int(m.group(3))
        rows.append(dict(file=f, pid=pid, stage=stage, date=date, seq=seq))

print(f"{len(rows)} files parsed, {len({r['pid'] for r in rows})} distinct patients")
print("  by stage:", dict(sorted(Counter(r['stage'] for r in rows).items())))

# the run order: date, then sequence within the day
for r in rows:
    dt = datetime.strptime(r["date"], "%m-%d-%Y") if int(r["date"].split("-")[0])<=12 \
         else datetime.strptime(r["date"], "%d-%m-%Y")
    r["order_key"] = (dt, r["seq"])
rows.sort(key=lambda r: r["order_key"])
for i,r in enumerate(rows): r["run_order"] = i

print("\nACQUISITION DATE AGAINST GROUP")
tab = defaultdict(Counter)
for r in rows: tab[r["date"]][r["stage"]] += 1
print(f"  {'date':12s} " + "".join(f"{('NoCKD' if s==0 else 'CKD'+str(s)):>7s}" for s in range(5)))
for d in sorted(tab, key=lambda x: datetime.strptime(x,"%m-%d-%Y")):
    print(f"  {d:12s} " + "".join(f"{tab[d].get(s,0):7d}" for s in range(5)))

# how much does run order alone separate each contrast?
from scipy.stats import mannwhitneyu
def order_auc(sel, pos):
    a = [r["run_order"] for r in rows if sel(r) and pos(r)]
    b = [r["run_order"] for r in rows if sel(r) and not pos(r)]
    if len(a)<5 or len(b)<5: return None
    u,_ = mannwhitneyu(a,b); v = u/(len(a)*len(b))
    return max(v,1-v), len(a), len(b)

print("\nAUC OBTAINABLE FROM RUN ORDER ALONE")
for name, sel, pos in [
    ("detect_any_ckd", lambda r: True,            lambda r: r["stage"]>0),
    ("progression",    lambda r: True,            lambda r: r["stage"]>=3),
    ("severity",       lambda r: r["stage"]>0,    lambda r: r["stage"]>=3),
    ("stage_1_vs_2",   lambda r: r["stage"] in (1,2), lambda r: r["stage"]==2)]:
    v = order_auc(sel,pos)
    if v: print(f"  {name:18s} {v[0]:.4f}   ({v[1]} vs {v[2]})")

# rebuild profiles keyed by the true patient id
X, pids, meta = [], [], []
for k,r in enumerate(rows,1):
    e = oms.MSExperiment(); oms.MzMLFile().load(str(r["file"]), e)
    mz  = np.concatenate([s.get_peaks()[0] for s in e.getSpectra()])
    ity = np.concatenate([s.get_peaks()[1] for s in e.getSpectra()])
    m = (mz>=MZ_MIN)&(mz<MZ_MAX)
    prof,_ = np.histogram(mz[m], bins=NBINS, range=(MZ_MIN,MZ_MAX), weights=ity[m])
    X.append(prof.astype(np.float32)); pids.append(r["pid"])
    meta.append({k2:v for k2,v in r.items() if k2 not in ("file","order_key")})
    if k % 60 == 0: print(f"  {k}/{len(rows)}")

np.save(OUT/"mz_profiles.npy", np.vstack(X))
json.dump(pids, open(OUT/"profile_pids.json","w"))
json.dump(meta, open(OUT/"metadata.json","w"), indent=1)
print(f"\nprofiles rebuilt: {np.vstack(X).shape}")

try:
    known = set(json.load(open("profiles/profile_pids.json")))
    print(f"overlap with the original cohort: {len(set(pids)&known)} of {len(set(pids))}")
except FileNotFoundError:
    pass

222 files parsed, 222 distinct patients
  by stage: {0: 37, 1: 28, 2: 80, 3: 70, 4: 7}

ACQUISITION DATE AGAINST GROUP
  date           NoCKD   CKD1   CKD2   CKD3   CKD4
  4-1-2025          12      0      0      0      0
  4-2-2025          19      0      0      0      0
  4-3-2025           6     28      0      0      0
  4-4-2025           0      0      0     18      0
  4-5-2025           0      0      0     37      0
  4-6-2025           0      0     15     15      0
  4-7-2025           0      0     35      0      0
  4-8-2025           0      0     30      0      7

AUC OBTAINABLE FROM RUN ORDER ALONE
  detect_any_ckd     1.0000   (185 vs 37)
  progression        0.5016   (77 vs 145)
  severity           0.6734   (77 vs 108)
  stage_1_vs_2       1.0000   (80 vs 28)
  60/222
  120/222
  180/222

profiles rebuilt: (222, 80000)
overlap with the original cohort: 145 of 222


In [5]:
import json
import numpy as np
from collections import Counter, defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold
from scipy.stats import mannwhitneyu

meta = json.load(open("second_cohort/metadata.json"))
dates = np.array([r["date"] for r in meta])
stage = np.array([r["stage"] for r in meta])
order = np.array([r["run_order"] for r in meta])

TASKS = {
 "detect_any_ckd": (np.ones(len(stage),bool),      stage>0),
 "progression":    (np.ones(len(stage),bool),      stage>=3),
 "severity":       (stage>0,                       stage>=3),
 "stage_1_vs_2":   (np.isin(stage,[1,2]),          stage==2),
}

print(f"{'task':18s} {'n':>5s} {'run order':>10s} {'date as category':>18s}  interpretation")
print("-"*84)
for name,(sel,pos) in TASKS.items():
    y = pos[sel].astype(int); d = dates[sel]; o = order[sel]
    if min(np.bincount(y)) < 5: continue

    u,_ = mannwhitneyu(o[y==1], o[y==0])
    v = u/((y==1).sum()*(y==0).sum()); rank_auc = max(v,1-v)

    X = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit_transform(d.reshape(-1,1))
    oof = np.zeros(len(y)); cnt = np.zeros(len(y))
    for tr,te in RepeatedStratifiedKFold(n_splits=5,n_repeats=10,
                                         random_state=42).split(X,y):
        m = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X[tr],y[tr])
        oof[te] += m.predict_proba(X[te])[:,1]; cnt[te]+=1
    cat_auc = roc_auc_score(y, oof/cnt)

    verd = ("entirely batch" if cat_auc>0.90 else
            "largely batch"  if cat_auc>0.75 else
            "partly batch"   if cat_auc>0.60 else "batch is weak here")
    print(f"{name:18s} {len(y):5d} {rank_auc:10.4f} {cat_auc:18.4f}  {verd}")

print("\nPURITY OF EACH ACQUISITION DAY  (progression: stage>=3 positive)")
by = defaultdict(Counter)
for r in meta: by[r["date"]][1 if r["stage"]>=3 else 0]+= 1
print(f"  {'date':12s} {'pos':>5s} {'neg':>5s}  {'purity':>7s}")
det = 0
for d in sorted(by):
    c = by[d]; n = c[0]+c[1]; p = max(c[0],c[1])/n
    if p == 1.0: det += n
    print(f"  {d:12s} {c[1]:5d} {c[0]:5d}  {p:7.0%}{'  <- fully determined' if p==1.0 else ''}")
print(f"\n  {det} of {len(meta)} patients have their class fixed by acquisition date alone.")

print("""
A rank-based baseline detects a monotonic trend: one class drifting later
than the other. It does not detect block structure, where each group is run
on its own day. Both are batch effects, and only the first is visible to the
patient-identifier test used elsewhere in this thesis.""")

task                   n  run order   date as category  interpretation
------------------------------------------------------------------------------------
detect_any_ckd       222     1.0000             0.9760  entirely batch
progression          222     0.5016             0.9552  entirely batch
severity             185     0.6734             0.9365  entirely batch
stage_1_vs_2         108     1.0000             1.0000  entirely batch

PURITY OF EACH ACQUISITION DAY  (progression: stage>=3 positive)
  date           pos   neg   purity
  4-1-2025         0    12     100%  <- fully determined
  4-2-2025         0    19     100%  <- fully determined
  4-3-2025         0    34     100%  <- fully determined
  4-4-2025        18     0     100%  <- fully determined
  4-5-2025        37     0     100%  <- fully determined
  4-6-2025        15    15      50%
  4-7-2025         0    35     100%  <- fully determined
  4-8-2025         7    30      81%

  155 of 222 patients have their class fixe

In [7]:

import json
import numpy as np
from pathlib import Path
from scipy.stats import mannwhitneyu, chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold

pids = json.load(open("profiles/profile_pids.json"))
num = {p: int(d) for p in pids if (d := "".join(c for c in p if c.isdigit()))}
NB   = 10                       # blocks of enrollment order

TASKS = ["detect_any_ckd","progression","severity_all","severity_nondiabetic",
         "positive_control_diabetes","matched_nondiabetic","detect_nondiabetic"]

print(f"{'task':26s} {'n':>5s} {'rank AUC':>9s} {'block AUC':>10s} {'chi2 p':>10s}  note")
print("-"*82)
for t in TASKS:
    f = Path(f"labels/labels_{t}.json")
    if not f.exists(): continue
    lab = json.load(open(f))["labels"]
    ids = np.array([num[p] for p in pids if p in lab and p in num])
    y   = np.array([lab[p] for p in pids if p in lab and p in num])
    if len(y) < 50 or min(np.bincount(y)) < 10: continue

    u,_ = mannwhitneyu(ids[y==1], ids[y==0])
    v = u/((y==1).sum()*(y==0).sum()); rank_auc = max(v,1-v)

    # split enrollment order into blocks and use block membership as a category
    edges = np.quantile(ids, np.linspace(0,1,NB+1))
    blk = np.clip(np.digitize(ids, edges[1:-1]), 0, NB-1)
    X = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit_transform(blk.reshape(-1,1))
    oof = np.zeros(len(y)); cnt = np.zeros(len(y))
    for tr,te in RepeatedStratifiedKFold(n_splits=5,n_repeats=10,
                                         random_state=42).split(X,y):
        m = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X[tr],y[tr])
        oof[te] += m.predict_proba(X[te])[:,1]; cnt[te]+=1
    blk_auc = roc_auc_score(y, oof/cnt)

    tab = np.zeros((NB,2))
    for b,c in zip(blk,y): tab[b,c] += 1
    keep = tab.sum(1) > 0
    p = chi2_contingency(tab[keep]+0.5)[1]

    note = ""
    if blk_auc - rank_auc > 0.05: note = "block structure the rank test misses"
    elif blk_auc > 0.70:          note = "strong batch association"
    print(f"{t:26s} {len(y):5d} {rank_auc:9.4f} {blk_auc:10.4f} {p:10.2e}  {note}")

print(f"""
The rank AUC is the patient-identifier baseline used throughout the thesis.
The block AUC uses the same information split into {NB} contiguous blocks and
treated as a category, which detects group-wise structure rather than drift.
Where the two agree, the confound is monotonic and the existing baseline is
adequate. Where the block AUC is materially higher, there is structure the
thesis has not so far accounted for.""")

task                           n  rank AUC  block AUC     chi2 p  note
----------------------------------------------------------------------------------
detect_any_ckd               596    0.7192     0.7151   8.88e-10  strong batch association
progression                  596    0.5768     0.5793   3.55e-04  
severity_all                 510    0.5352     0.5821   1.04e-04  
severity_nondiabetic         359    0.5187     0.5467   1.39e-01  
positive_control_diabetes    596    0.6431     0.6401   8.45e-14  
matched_nondiabetic           92    0.5019     0.1805   1.00e+00  
detect_nondiabetic           405    0.6947     0.6818   1.12e-04  

The rank AUC is the patient-identifier baseline used throughout the thesis.
The block AUC uses the same information split into 10 contiguous blocks and
treated as a category, which detects group-wise structure rather than drift.
Where the two agree, the confound is monotonic and the existing baseline is
adequate. Where the block AUC is materially hig